## メインパート

In [85]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [86]:
!pip install category_encoders -q

In [115]:
import warnings
warnings.filterwarnings("ignore", category=UserWarning)

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.preprocessing import LabelEncoder, OneHotEncoder, StandardScaler
from category_encoders import HashingEncoder
from IPython.display import display, Markdown
from tabulate import tabulate
from sklearn.metrics import roc_auc_score, accuracy_score, log_loss
from sklearn.neural_network import MLPClassifier
print(torch.cuda.is_available())

False


__タスク_1 - データのダウンロードと前処理__

In [88]:
df = pd.read_csv("drive/MyDrive/Colab_Notebooks/s21_ML8/training.csv")
df.head(2)

,RefId,IsBadBuy,PurchDate,Auction,VehYear,VehicleAge,Make,Model,Trim,SubModel,...,MMRCurrentRetailAveragePrice,MMRCurrentRetailCleanPrice,PRIMEUNIT,AUCGUART,BYRNO,VNZIP1,VNST,VehBCost,IsOnlineSale,WarrantyCost
0,1,0,12/7/2009,ADESA,2006,3,MAZDA,MAZDA3,i,4D SEDAN I,...,11597.0,12409.0,NaN,NaN,21973,33619,FL,7100.0,0,1113
1,2,0,12/7/2009,ADESA,2004,5,DODGE,1500 RAM PICKUP 2WD,ST,QUAD CAB 4.7L SLT,...,11374.0,12791.0,NaN,NaN,19638,33619,FL,7600.0,0,1053


ギャップを埋める

In [89]:
df.isna().sum()[lambda x: x > 0].sort_values(ascending=False)

,0
AUCGUART,69564
PRIMEUNIT,69564
WheelType,3174
WheelTypeID,3169
Trim,2360
MMRCurrentAuctionAveragePrice,315
MMRCurrentAuctionCleanPrice,315
MMRCurrentRetailAveragePrice,315
MMRCurrentRetailCleanPrice,315
MMRAcquisitionAuctionCleanPrice,18


In [90]:
cat_features_with_missing = ['Trim', 'SubModel', 'Color', 'Size', 'Transmission', 'WheelType', 'Nationality', 'Size', 'TopThreeAmericanName', 'AUCGUART', 'PRIMEUNIT']
for col in cat_features_with_missing:
    df[col] = df[col].fillna("missing")

In [91]:
numeric_features = ['WheelTypeID', 'MMRAcquisitionAuctionAveragePrice', 'MMRAcquisitionAuctionCleanPrice', 'MMRAcquisitionRetailAveragePrice', 'MMRAcquisitonRetailCleanPrice', 'MMRCurrentAuctionAveragePrice', 'MMRCurrentAuctionCleanPrice', 'MMRCurrentRetailAveragePrice', 'MMRCurrentRetailCleanPrice']
for col in numeric_features:
    median_val = df[col].median()
    df[col] = df[col].fillna(median_val)

In [92]:
df.isna().sum()[lambda x: x > 0].sort_values(ascending=False)

,0


日付別にデータを .33/.33/.33 の比率で分割

In [93]:
date_threshold_1 = pd.to_datetime("2009-09-15")
date_threshold_2 = pd.to_datetime("2010-05-15")
df['PurchDate'] = pd.to_datetime(df['PurchDate'])

df_train = df[df['PurchDate'] < date_threshold_1]
df_val = df[(df['PurchDate'] >= date_threshold_1) & (df['PurchDate'] < date_threshold_2)]
df_test = df[df['PurchDate'] >= date_threshold_2]

print(f"train_data: {df_train.shape[0]} x {df_train.shape[1]}, percentile_shape : {int((df_train.shape[0]/df.shape[0])*100)}%")
print(f"val_data:   {df_val.shape[0]} x {df_val.shape[1]}, percentile_shape : {int(df_val.shape[0]/df.shape[0]*100)}%")
print(f"test_data:  {df_test.shape[0]} x {df_test.shape[1]}, percentile_shape : {int(df_test.shape[0]/df.shape[0]*100)}%")

train_data: 24232 x 34, percentile_shape : 33%
val_data:   24486 x 34, percentile_shape : 33%
test_data:  24265 x 34, percentile_shape : 33%


In [94]:
print('Checking:')
if max(df_train['PurchDate']) < max(df_val['PurchDate']) < min(df_test['PurchDate']):
    print("train < val < test")
else:
    print("Uvays")

df_train = df_train.drop(['PurchDate', 'RefId'], axis=1)
df_val = df_val.drop(['PurchDate', 'RefId'], axis=1)
df_test = df_test.drop(['PurchDate', 'RefId'], axis=1)

Checking:
train < val < test


In [95]:
df_cat_features = df_train.select_dtypes(include='object')
cat_unique_counts = df_cat_features.nunique().sort_values(ascending=False)
print(cat_unique_counts)

Model                   801
SubModel                663
Trim                    123
Make                     31
VNST                     29
Color                    17
Size                     12
TopThreeAmericanName      4
Nationality               4
WheelType                 4
Auction                   3
Transmission              3
PRIMEUNIT                 2
AUCGUART                  2
dtype: int64


Hashing encoding : ['Model', 'SubModel', 'Trim'] \
Frequency encoding : ['Make', 'VNST', 'Color', 'Size'] \
OneHotEncoder : ['TopThreeAmericanName', 'Nationality', 'WheelType', 'Auction', 'Transmission', 'PRIMEUNIT', 'AUCGUART']

In [96]:
features_for_OHE = ['TopThreeAmericanName', 'Nationality', 'WheelType', 'Auction', 'Transmission', 'PRIMEUNIT', 'AUCGUART']

ohe = OneHotEncoder(handle_unknown='ignore', sparse_output=False)
ohe.fit(df_train[features_for_OHE])

X_train = pd.DataFrame(ohe.transform(df_train[features_for_OHE]), columns=ohe.get_feature_names_out(features_for_OHE), index=df_train.index)
X_val   = pd.DataFrame(ohe.transform(df_val[features_for_OHE]),   columns=ohe.get_feature_names_out(features_for_OHE), index=df_val.index)
X_test  = pd.DataFrame(ohe.transform(df_test[features_for_OHE]),  columns=ohe.get_feature_names_out(features_for_OHE), index=df_test.index)

df_train = df_train.drop(columns=features_for_OHE).join(X_train)
df_val   = df_val.drop(columns=features_for_OHE).join(X_val)
df_test  = df_test.drop(columns=features_for_OHE).join(X_test)

In [97]:
print(f"train_data: {df_train.shape[0]} x {df_train.shape[1]}")
print(f"val_data:   {df_val.shape[0]} x {df_val.shape[1]}")
print(f"test_data:  {df_test.shape[0]} x {df_test.shape[1]}")

train_data: 24232 x 47
val_data:   24486 x 47
test_data:  24265 x 47


In [98]:
cat_features = df_train.select_dtypes(include='object')
freq_features = ['Make', 'Color', 'Size', 'VNST']
for col in freq_features:
    freq = df_train[col].value_counts(normalize=True)
    df_train[col] = df_train[col].map(freq)
    df_val[col] = df_val[col].map(freq).fillna(0)
    df_test[col] = df_test[col].map(freq).fillna(0)

hash_cols = ['Trim', 'Model', 'SubModel']
hash_enc = HashingEncoder(cols=hash_cols, n_components=8)
hash_enc.fit(df_train[hash_cols])

df_train_hash = hash_enc.transform(df_train[hash_cols])
df_val_hash = hash_enc.transform(df_val[hash_cols])
df_test_hash = hash_enc.transform(df_test[hash_cols])

df_train = df_train.drop(columns=hash_cols).join(df_train_hash)
df_val = df_val.drop(columns=hash_cols).join(df_val_hash)
df_test = df_test.drop(columns=hash_cols).join(df_test_hash)

In [99]:
print(f"train_data: {df_train.shape[0]} x {df_train.shape[1]}")
print(f"val_data:   {df_val.shape[0]} x {df_val.shape[1]}")
print(f"test_data:  {df_test.shape[0]} x {df_test.shape[1]}")

train_data: 24232 x 52
val_data:   24486 x 52
test_data:  24265 x 52


In [100]:
X_train = df_train.drop('IsBadBuy', axis=1)
X_val = df_val.drop('IsBadBuy', axis=1)
X_test = df_test.drop('IsBadBuy', axis=1)

y_train = df_train['IsBadBuy']
y_val = df_val['IsBadBuy']
y_test = df_test['IsBadBuy']

__タスク_2 - 実装MLPクラス__

In [101]:
class MLP:
    def __init__(self, n_hidden=100, activation=np.tanh, random_state=42):
        self.n_hidden = n_hidden
        self.activation = activation
        self.rng = np.random.RandomState(random_state)

    def _init_weights(self, n_features):
        self.W1 = 0.01 * self.rng.randn(n_features, self.n_hidden)
        self.b1 = np.zeros((1, self.n_hidden))
        self.W2 = 0.01 * self.rng.randn(self.n_hidden, 1)
        self.b2 = np.zeros((1, 1))

    def _sigmoid(self, z): return 1 / (1 + np.exp(-z))

    def _activation_grad(self, z, eps=1e-6):
        if self.activation == np.tanh: return 1 - np.tanh(z)**2
        elif self.activation == np.arctan: return 1 / (1 + z**2)
        else: return (self.activation(z+eps) - self.activation(z-eps)) / (2*eps)

    def fit(self, X, y, X_val=None, y_val=None, epochs=20, lr=0.01, batch_size=32, eps=1e-9):
        n_samples, n_features = X.shape
        self._init_weights(n_features)

        y = y.reshape(-1, 1)

        for epoch in range(epochs):
            indices = np.random.permutation(n_samples)
            X_shuff, y_shuff = X[indices], y[indices]

            for start in range(0, n_samples, batch_size):
                end = start + batch_size
                xb, yb = X_shuff[start:end], y_shuff[start:end]

                z1 = xb @ self.W1 + self.b1
                a1 = self.activation(z1)
                z2 = a1 @ self.W2 + self.b2
                probs = self._sigmoid(z2)

                loss = -np.mean(yb*np.log(probs+eps) + (1-yb)*np.log(1-probs+eps))

                dz2 = probs - yb
                dW2 = a1.T @ dz2 / batch_size
                db2 = np.mean(dz2, axis=0, keepdims=True)

                da1 = dz2 @ self.W2.T
                dz1 = da1 * self._activation_grad(z1)
                dW1 = xb.T @ dz1 / batch_size
                db1 = np.mean(dz1, axis=0, keepdims=True)

                self.W1 -= lr * dW1
                self.b1 -= lr * db1
                self.W2 -= lr * dW2
                self.b2 -= lr * db2

            if X_val is not None and y_val is not None:
                y_val_pred_proba = self.predict_proba(X_val)
                val_loss = -np.mean(y_val.reshape(-1,1)*np.log(y_val_pred_proba+eps) + (1-y_val.reshape(-1,1))*np.log(1-y_val_pred_proba+eps))
                val_acc = np.mean(self.predict(X_val) == y_val)

                print(f"Epoch {epoch+1:02d}/{epochs}    "
                      f"loss={loss:.4f}    "
                      f"val_loss={val_loss:.4f}    "
                      f"val_acc={val_acc:.4f}")

    def predict_proba(self, X):
        z1 = X @ self.W1 + self.b1
        a1 = self.activation(z1)
        z2 = a1 @ self.W2 + self.b2
        return self._sigmoid(z2)

    def predict(self, X, threshold=0.5):
        return (self.predict_proba(X) >= threshold).astype(int).ravel()

__タスク3 - MLPクラスの使用__

In [102]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

y_train_np = np.asarray(y_train)
y_val_np   = np.asarray(y_val)

In [103]:
custom_model = MLP(n_hidden=200,
                   activation=np.tanh)

custom_model.fit(X_train_scaled, y_train_np,
                 X_val=X_val_scaled, y_val=y_val_np,
                 epochs=20,
                 lr=0.005)

Epoch 01/20    loss=0.3143    val_loss=0.4308    val_acc=0.8696
Epoch 02/20    loss=0.2290    val_loss=0.3660    val_acc=0.8696
Epoch 03/20    loss=0.1467    val_loss=0.3422    val_acc=0.8738
Epoch 04/20    loss=0.6070    val_loss=0.3354    val_acc=0.8828
Epoch 05/20    loss=0.4523    val_loss=0.3354    val_acc=0.8815
Epoch 06/20    loss=0.4114    val_loss=0.3363    val_acc=0.8805
Epoch 07/20    loss=0.3613    val_loss=0.3372    val_acc=0.8802
Epoch 08/20    loss=0.4330    val_loss=0.3382    val_acc=0.8803
Epoch 09/20    loss=0.1097    val_loss=0.3383    val_acc=0.8803
Epoch 10/20    loss=0.1046    val_loss=0.3385    val_acc=0.8800
Epoch 11/20    loss=0.3078    val_loss=0.3385    val_acc=0.8802
Epoch 12/20    loss=0.1343    val_loss=0.3386    val_acc=0.8801
Epoch 13/20    loss=0.2970    val_loss=0.3388    val_acc=0.8801
Epoch 14/20    loss=0.2839    val_loss=0.3391    val_acc=0.8801
Epoch 15/20    loss=0.0951    val_loss=0.3392    val_acc=0.8802
Epoch 16/20    loss=0.2852    val_loss=0

In [104]:
def gini_score(y_true, y_prob):
    auc = roc_auc_score(y_true, y_prob)
    return 2 * auc - 1

In [105]:
y_val_pred = custom_model.predict_proba(X_val_scaled)
display(Markdown(f"Gini score:  **{gini_score(y_val, y_val_pred):.3f}**"))

Gini score:  **0.474**

__タスク4 - sklearn MLP分類器の使用__

In [106]:
sklearn_model = MLPClassifier(hidden_layer_sizes=(200,),
                              activation="tanh",
                              solver="sgd",
                              learning_rate_init=0.01,
                              alpha=0.005,
                              max_iter=1,
                              warm_start=True,
                              random_state=42)

n_epochs = 20
for epoch in range(n_epochs):
    sklearn_model.fit(X_train_scaled, y_train_np)

    train_loss = sklearn_model.loss_
    y_val_pred_proba = sklearn_model.predict_proba(X_val_scaled)[:, 1]
    val_loss = log_loss(y_val_np, y_val_pred_proba)
    val_acc = accuracy_score(y_val_np, (y_val_pred_proba > 0.5).astype(int))

    print(f"Epoch {epoch+1:02d}/{n_epochs}    "
          f"loss={train_loss:.4f}    "
          f"val_loss={val_loss:.4f}    "
          f"val_acc={val_acc:.4f}")

Epoch 01/20    loss=0.3441    val_loss=0.3411    val_acc=0.8813
Epoch 02/20    loss=0.2992    val_loss=0.3410    val_acc=0.8806
Epoch 03/20    loss=0.2985    val_loss=0.3404    val_acc=0.8805
Epoch 04/20    loss=0.2980    val_loss=0.3398    val_acc=0.8806
Epoch 05/20    loss=0.2975    val_loss=0.3393    val_acc=0.8805
Epoch 06/20    loss=0.2970    val_loss=0.3389    val_acc=0.8804
Epoch 07/20    loss=0.2966    val_loss=0.3385    val_acc=0.8804
Epoch 08/20    loss=0.2962    val_loss=0.3381    val_acc=0.8803
Epoch 09/20    loss=0.2958    val_loss=0.3377    val_acc=0.8804
Epoch 10/20    loss=0.2953    val_loss=0.3374    val_acc=0.8805
Epoch 11/20    loss=0.2949    val_loss=0.3370    val_acc=0.8804
Epoch 12/20    loss=0.2945    val_loss=0.3367    val_acc=0.8805
Epoch 13/20    loss=0.2941    val_loss=0.3364    val_acc=0.8804
Epoch 14/20    loss=0.2937    val_loss=0.3361    val_acc=0.8804
Epoch 15/20    loss=0.2933    val_loss=0.3358    val_acc=0.8805
Epoch 16/20    loss=0.2929    val_loss=0

In [107]:
y_val_pred = sklearn_model.predict_proba(X_val_scaled)[:, 1]
display(Markdown(f"Gini score:  **{gini_score(y_val, y_val_pred):.3f}**"))

Gini score:  **0.484**

__タスク5 - 異なる活性化関数の使用__

In [108]:
def sigmoid(z): return 1 / (1 + np.exp(-z))

sigmoid_model = MLP(n_hidden=200,
                    activation=sigmoid)

sigmoid_model.fit(X_train_scaled, y_train_np,
                  X_val=X_val_scaled, y_val=y_val_np,
                  epochs=20,
                  lr=0.005)

Epoch 01/20    loss=0.3734    val_loss=0.3866    val_acc=0.8696
Epoch 02/20    loss=0.3775    val_loss=0.3877    val_acc=0.8696
Epoch 03/20    loss=0.3687    val_loss=0.3848    val_acc=0.8696
Epoch 04/20    loss=0.1246    val_loss=0.3838    val_acc=0.8696
Epoch 05/20    loss=0.6045    val_loss=0.3820    val_acc=0.8696
Epoch 06/20    loss=0.6244    val_loss=0.3832    val_acc=0.8696
Epoch 07/20    loss=0.3837    val_loss=0.3806    val_acc=0.8696
Epoch 08/20    loss=0.1194    val_loss=0.3768    val_acc=0.8696
Epoch 09/20    loss=0.3189    val_loss=0.3755    val_acc=0.8696
Epoch 10/20    loss=0.1343    val_loss=0.3712    val_acc=0.8696
Epoch 11/20    loss=0.6376    val_loss=0.3709    val_acc=0.8696
Epoch 12/20    loss=0.5778    val_loss=0.3705    val_acc=0.8696
Epoch 13/20    loss=0.1146    val_loss=0.3637    val_acc=0.8696
Epoch 14/20    loss=0.4164    val_loss=0.3645    val_acc=0.8696
Epoch 15/20    loss=0.0987    val_loss=0.3564    val_acc=0.8696
Epoch 16/20    loss=0.1346    val_loss=0

In [109]:
def relu(z): return np.maximum(0, z)

relu_model = MLP(n_hidden=200,
                 activation=relu)

relu_model.fit(X_train_scaled, y_train_np,
               X_val=X_val_scaled, y_val=y_val_np,
               epochs=20,
               lr=0.005)

Epoch 01/20    loss=0.2737    val_loss=0.4197    val_acc=0.8696
Epoch 02/20    loss=0.1883    val_loss=0.3739    val_acc=0.8696
Epoch 03/20    loss=0.1537    val_loss=0.3667    val_acc=0.8696
Epoch 04/20    loss=0.5352    val_loss=0.3644    val_acc=0.8696
Epoch 05/20    loss=0.5350    val_loss=0.3618    val_acc=0.8696
Epoch 06/20    loss=0.1307    val_loss=0.3584    val_acc=0.8696
Epoch 07/20    loss=0.0795    val_loss=0.3534    val_acc=0.8696
Epoch 08/20    loss=0.0756    val_loss=0.3463    val_acc=0.8697
Epoch 09/20    loss=0.1945    val_loss=0.3396    val_acc=0.8712
Epoch 10/20    loss=0.5809    val_loss=0.3357    val_acc=0.8840
Epoch 11/20    loss=0.3213    val_loss=0.3349    val_acc=0.8815
Epoch 12/20    loss=0.0936    val_loss=0.3349    val_acc=0.8814
Epoch 13/20    loss=0.5077    val_loss=0.3352    val_acc=0.8817
Epoch 14/20    loss=0.5999    val_loss=0.3351    val_acc=0.8814
Epoch 15/20    loss=0.1017    val_loss=0.3352    val_acc=0.8812
Epoch 16/20    loss=0.3440    val_loss=0

In [110]:
def cosine(z): return np.cos(z)

cosine_model = MLP(n_hidden=200,
                   activation=cosine)

cosine_model.fit(X_train_scaled, y_train_np,
                 X_val=X_val_scaled, y_val=y_val_np,
                 epochs=20,
                 lr=0.005)

Epoch 01/20    loss=0.1596    val_loss=0.3875    val_acc=0.8697
Epoch 02/20    loss=0.6409    val_loss=0.3884    val_acc=0.8697
Epoch 03/20    loss=0.1225    val_loss=0.3883    val_acc=0.8697
Epoch 04/20    loss=0.3767    val_loss=0.3875    val_acc=0.8697
Epoch 05/20    loss=0.1093    val_loss=0.3910    val_acc=0.8696
Epoch 06/20    loss=0.1203    val_loss=0.3885    val_acc=0.8696
Epoch 07/20    loss=0.6008    val_loss=0.3880    val_acc=0.8696
Epoch 08/20    loss=0.1284    val_loss=0.3874    val_acc=0.8696
Epoch 09/20    loss=0.6360    val_loss=0.3874    val_acc=0.8696
Epoch 10/20    loss=0.3768    val_loss=0.3896    val_acc=0.8696
Epoch 11/20    loss=0.3791    val_loss=0.3893    val_acc=0.8696
Epoch 12/20    loss=0.6279    val_loss=0.3863    val_acc=0.8696
Epoch 13/20    loss=0.1272    val_loss=0.3857    val_acc=0.8696
Epoch 14/20    loss=0.3769    val_loss=0.3843    val_acc=0.8696
Epoch 15/20    loss=0.3809    val_loss=0.3821    val_acc=0.8696
Epoch 16/20    loss=0.1256    val_loss=0

In [111]:
y_val_pred_sigmoid = sigmoid_model.predict_proba(X_val_scaled)
y_val_pred_relu = relu_model.predict_proba(X_val_scaled)
y_val_pred_cosine = cosine_model.predict_proba(X_val_scaled)

display(Markdown(f"Gini score for sigmoid: **{gini_score(y_val, y_val_pred_sigmoid):.3f}**"))
display(Markdown(f"Gini score for relu:    **{gini_score(y_val, y_val_pred_relu):.3f}**"))
display(Markdown(f"Gini score for cosine:  **{gini_score(y_val, y_val_pred_cosine):.3f}**"))

Gini score for sigmoid: **0.456**

Gini score for relu:    **0.477**

Gini score for cosine:  **0.216**

In [118]:
y_train_pred_tanh     = custom_model.predict_proba(X_train_scaled)
y_train_pred_sigmoid  = sigmoid_model.predict_proba(X_train_scaled)
y_train_pred_relu     = relu_model.predict_proba(X_train_scaled)
y_train_pred_cosine   = cosine_model.predict_proba(X_train_scaled)

display(Markdown(f"train Gini score for tanh:  **{gini_score(y_train, y_train_pred_tanh):.3f}**"))
display(Markdown(f"train Gini score for sigmoid: **{gini_score(y_train, y_train_pred_sigmoid):.3f}**"))
display(Markdown(f"train Gini score for relu:    **{gini_score(y_train, y_train_pred_relu):.3f}**"))
display(Markdown(f"train Gini score for cosine:  **{gini_score(y_train, y_train_pred_cosine):.3f}**"))

train Gini score for tanh:  **0.505**

train Gini score for sigmoid: **0.483**

train Gini score for relu:    **0.525**

train Gini score for cosine:  **0.346**

__タスク_6 - PytorchによるMLP__

In [112]:
class TorchMLP(nn.Module):
    def __init__(self, n_features, n_hidden=100, activation="tanh"):
        super(TorchMLP, self).__init__()

        self.fc1 = nn.Linear(n_features, n_hidden)
        self.fc2 = nn.Linear(n_hidden, 1)

        if activation == "tanh": self.activation = nn.Tanh()
        elif activation == "relu": self.activation = nn.ReLU()
        else: raise ValueError("Unsupported activation")

        self.out_act = nn.Sigmoid()
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        self.to(self.device)

    def forward(self, x):
        x = self.activation(self.fc1(x))
        x = self.out_act(self.fc2(x))
        return x

    def fit(self, X_train, y_train, X_val=None, y_val=None,
            lr=0.01, epochs=20, batch_size=32):

        X_train_t = torch.tensor(X_train, dtype=torch.float32).to(self.device)
        y_train_t = torch.tensor(y_train.reshape(-1,1), dtype=torch.float32).to(self.device)

        if X_val is not None and y_val is not None:
            X_val_t = torch.tensor(X_val, dtype=torch.float32).to(self.device)
            y_val_t = torch.tensor(y_val.reshape(-1,1), dtype=torch.float32).to(self.device)

        criterion = nn.BCELoss()
        optimizer = optim.SGD(self.parameters(), lr=lr)

        n_samples = X_train.shape[0]

        for epoch in range(epochs):
            perm = torch.randperm(n_samples)

            for i in range(0, n_samples, batch_size):
                idx = perm[i:i+batch_size]
                xb, yb = X_train_t[idx], y_train_t[idx]

                optimizer.zero_grad()
                preds = self(xb)
                loss = criterion(preds, yb)
                loss.backward()
                optimizer.step()

            if X_val is not None and y_val is not None:
                with torch.no_grad():
                    val_probs = self(X_val_t).cpu().numpy().ravel()
                    val_preds = (val_probs >= 0.5).astype(int)
                    val_acc = accuracy_score(y_val, val_preds)
                    val_loss = log_loss(y_val, val_probs)

                print(f"Epoch {epoch+1:02d}/{epochs}    "
                      f"train_loss={loss.item():.4f}    "
                      f"val_loss={val_loss:.4f}    "
                      f"val_acc={val_acc:.4f}")

    def predict_proba(self, X):
        self.eval()
        X_t = torch.tensor(X, dtype=torch.float32).to(self.device)
        with torch.no_grad():
            probs = self(X_t).cpu().numpy().ravel()
        return np.vstack([1 - probs, probs]).T

    def predict(self, X):
        probs = self.predict_proba(X)[:, 1]
        return (probs >= 0.5).astype(int)

In [113]:
torch_model = TorchMLP(n_hidden=200, n_features=X_train_scaled.shape[1])

torch_model.fit(X_train_scaled, y_train_np,
                X_val=X_val_scaled, y_val=y_val_np,
                epochs=20,
                lr=0.005)

Epoch 01/20    train_loss=0.1997    val_loss=0.3544    val_acc=0.8821
Epoch 02/20    train_loss=0.3668    val_loss=0.3353    val_acc=0.8814
Epoch 03/20    train_loss=0.3948    val_loss=0.3345    val_acc=0.8808
Epoch 04/20    train_loss=0.3076    val_loss=0.3350    val_acc=0.8806
Epoch 05/20    train_loss=0.2966    val_loss=0.3352    val_acc=0.8807
Epoch 06/20    train_loss=0.3222    val_loss=0.3354    val_acc=0.8804
Epoch 07/20    train_loss=0.1178    val_loss=0.3355    val_acc=0.8805
Epoch 08/20    train_loss=0.3017    val_loss=0.3354    val_acc=0.8806
Epoch 09/20    train_loss=0.4091    val_loss=0.3351    val_acc=0.8806
Epoch 10/20    train_loss=0.2654    val_loss=0.3356    val_acc=0.8807
Epoch 11/20    train_loss=0.3822    val_loss=0.3353    val_acc=0.8805
Epoch 12/20    train_loss=0.0780    val_loss=0.3352    val_acc=0.8805
Epoch 13/20    train_loss=0.0847    val_loss=0.3350    val_acc=0.8805
Epoch 14/20    train_loss=0.4377    val_loss=0.3350    val_acc=0.8804
Epoch 15/20    train

In [114]:
y_val_pred_proba = torch_model.predict_proba(X_val_scaled)[:, 1]
y_train_pred_proba = torch_model.predict_proba(X_train_scaled)[:, 1]
display(Markdown(f"Gini score:  **{gini_score(y_val_np, y_val_pred_proba):.3f}**"))
display(Markdown(f"Gini score:  **{gini_score(y_train_np, y_train_pred_proba):.3f}**"))

Gini score:  **0.477**

Gini score:  **0.513**

__タスク7 - 最適なモデルの選択__

In [120]:
data = {
    "N_hidden": ["200", "200", "200", "200"],
    "Activation": ["tanh", "sigmoid", "relu", "cosine"],
    "Epochs": ["20", "20", "20", "20"],
    "Learning_rate": ["0.005", "0.005", "0.005", "0.005"],
    "Train Gini score": ["0.505", "0.483", "0.525", "0.346"],
    "Val Gini score": ["0.474", "0.456", "0.477", "0.216"]
}

df = pd.DataFrame(data)
print(tabulate(df, headers='keys', tablefmt='fancy_grid', showindex=False, stralign='center', numalign='center'))

╒════════════╤══════════════╤══════════╤═════════════════╤════════════════════╤══════════════════╕
│  N_hidden  │  Activation  │  Epochs  │  Learning_rate  │  Train Gini score  │  Val Gini score  │
╞════════════╪══════════════╪══════════╪═════════════════╪════════════════════╪══════════════════╡
│    200     │     tanh     │    20    │      0.005      │       0.505        │      0.474       │
├────────────┼──────────────┼──────────┼─────────────────┼────────────────────┼──────────────────┤
│    200     │   sigmoid    │    20    │      0.005      │       0.483        │      0.456       │
├────────────┼──────────────┼──────────┼─────────────────┼────────────────────┼──────────────────┤
│    200     │     relu     │    20    │      0.005      │       0.525        │      0.477       │
├────────────┼──────────────┼──────────┼─────────────────┼────────────────────┼──────────────────┤
│    200     │    cosine    │    20    │      0.005      │       0.346        │      0.216       │
╘═════════

## ボーナス部分

__タスク - Adamアルゴリズムの実装__